In [1]:
import numpy as np
import pandas as pd
import psycopg
from sqlalchemy import create_engine 
import dotenv
import os

In [2]:
dotenv.load_dotenv()

False

In [3]:
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD")

In [5]:
games = pd.read_csv("data/games.csv")
teams = pd.read_csv("data/teams.csv")
players = pd.read_csv("data/players.csv")
team_game = pd.read_csv("data/team_game.csv")
player_game = pd.read_csv("data/player_game.csv")

#### Step 1: Connect to our Postgres Server (running via Docker), Create an empty NBA database

In [7]:
dbserver = psycopg.connect(
    host="localhost",
    port=5432,
    dbname="postgres",
    user="postgres",
    password=POSTGRES_PASSWORD
)

dbserver.autocommit = True

In [8]:
cursor = dbserver.cursor()

In [12]:
try:
    cursor.execute("CREATE DATABASE nba;")
except:
    cursor.execute("DROP DATABASE nba;")
    cursor.execute("CREATE DATABASE nba;")

#### Step 2: Put our data into that database

First we will create a SQL Alchemy engine -- a shortcut to the NBA database we just made

In [15]:
dbms = "postgresql"
package = "psycopg"
user = "postgres"
password = POSTGRES_PASSWORD
host = "localhost"
port = 5432
database = "nba"

engine = create_engine(f"{dbms}+{package}://{user}:{password}@{host}:{port}/{database}")

In [16]:
games.to_sql("games", con=engine, index=False, chunksize=1000, if_exists="replace")
players.to_sql("players", con=engine, index=False, chunksize=1000, if_exists="replace")
teams.to_sql("teams", con=engine, index=False, chunksize=1000, if_exists="replace")
player_game.to_sql("player_game", con=engine, index=False, chunksize=1000, if_exists="replace")
team_game.to_sql("team_game", con=engine, index=False, chunksize=1000, if_exists="replace")

-3

#### Step 2: SQL it like crazy

In [19]:
myquery = """
SELECT *
FROM games
WHERE home_team_id = 18

"""

pd.read_sql_query(myquery, con=engine)

,game_id,game_date,home_team_id,away_team_id,season_type
0,401809234.0,2025-10-22,18.0,5.0,Regular Season
1,401809238.0,2025-12-25,18.0,5.0,Regular Season
2,401809249.0,2026-01-19,18.0,6.0,Regular Season
3,401809781.0,2025-11-14,18.0,14.0,Regular Season
4,401809816.0,2025-11-28,18.0,15.0,Regular Season
5,401809839.0,2025-12-16,18.0,24.0,Regular Season
6,401809945.0,2025-10-24,18.0,2.0,Regular Season
7,401810008.0,2025-11-02,18.0,4.0,Regular Season
8,401810014.0,2025-11-03,18.0,27.0,Regular Season
9,401810030.0,2025-11-05,18.0,16.0,Regular Season


In [ ]:
# All player that score 50

In [27]:
myquery = """
SELECT pg.player_id,
    SUM(pg.pts + pg.fg3m - pg.fga + 2 * pg.fgm - pg.fta + pg.ftm + pg.oreb + pg.dreb + 2 * pg.ast + 4 * pg.stl + 4 * pg.blk - 2 * pg.tov) AS mvp_score
FROM player_game pg
INNER JOIN games g
    ON pg.game_id = g.game_id
INNER JOIN players p
    ON pg.player_id = p.player_id
WHERE g.season_type = 'Regular Season'
GROUP BY pg.player_id, p.player_name
ORDER BY mvp_score DESC
"""

pd.read_sql_query(myquery, con=engine)


,player_id,mvp_score
0,3112335.0,4310.0
1,3945274.0,3785.0
2,4278073.0,3773.0
3,4431678.0,3652.0
4,5104157.0,3579.0
...,...,...
577,5242496.0,1.0
578,4702785.0,1.0
579,4597865.0,1.0
580,4590360.0,1.0
